# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library following best practices for referencing all structures by their `@id` fields.

### Dataset Source
The dataset is defined by a Croissant schema and can be accessed at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` and required libraries are installed
!pip install --quiet mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset's Croissant metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not as a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their corresponding `@id` values.

**Note:** All entities (record sets, fields, columns) are referenced by their `@id` fields.

In [ ]:
# List available record sets and their @ids
print("List of record sets in the dataset:")
for record_set in dataset.record_sets:
    print(f"- Name: {record_set.name}\n  @id: {record_set.id}")

# Let's pick the main record set (usually first, or as appropriate).
main_record_set = dataset.record_sets[0]  # assuming main table is the first one
main_record_set_id = main_record_set.id

print(f"\nFields for main record set '@id': {main_record_set_id} ({main_record_set.name}):")
for field in main_record_set.fields:
    print(f"- Field Name: {field.name}, @id: {field.id}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from the selected record set into a pandas DataFrame.

**Use `@id` fields for reference.**

In [ ]:
# Extract data from all record sets and keep in a dict for flexible access
dataframes = dict()

for record_set in dataset.record_sets:
    recs = list(dataset.records(record_set=record_set.id))  # Use @id
    df = pd.DataFrame(recs)
    dataframes[record_set.id] = df

# Display columns for main record set
print(f"Main record set columns (@id={main_record_set.id}):")
print(dataframes[main_record_set.id].columns.tolist())

# Preview first few records
dataframes[main_record_set.id].head()

## 4. Exploratory Data Analysis (EDA)
Let's filter and process the data using field `@id`s.

We'll:
1. Select a numeric field by its `@id`
2. Filter records based on a threshold
3. Normalize the numeric field
4. Group by a categorical field (also by `@id`)

In [ ]:
# For demonstration, inspect main_record_set fields for a numeric one (e.g. 'Age_at_Second_CRC')

# Find a numeric field (float/int)
numeric_field = None
numeric_field_id = None
for field in main_record_set.fields:
    if 'Float' in field.data_type or 'Integer' in field.data_type or 'Number' in field.data_type:
        numeric_field = field.name
        numeric_field_id = field.id
        break
        
print(f"Numeric field selected: {numeric_field} (@id={numeric_field_id})")

# Pick a categorical/group field for grouping (e.g. 'Sex' or similar)
group_field = None
group_field_id = None
for field in main_record_set.fields:
    # Pick first non-numeric text field
    if (('Text' in field.data_type or 'String' in field.data_type) \
        and (field.id != numeric_field_id)):
        group_field = field.name
        group_field_id = field.id
        break
        
print(f"Group/categorical field selected: {group_field} (@id={group_field_id})")

# Prepare DataFrame and clean numeric column
df = dataframes[main_record_set.id].copy()

# Ensure the numeric column is float
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):\n", filtered_df[[numeric_field_id]].head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:\n", 
      filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical field if it exists
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships using the selected fields.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(8,5))
df[numeric_field_id].dropna().hist(bins=15, color='lightblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field} ({numeric_field_id})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group field present, boxplot
if group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² colorectal cancer dataset using `mlcroissant`.
- Identified record sets, fields, and operated strictly by their `@id` fields.
- Extracted tabular data and performed exploratory analysis on a selected numeric field.
- Explored data distributions and possible group differences visually.

This structured workflow facilitates reproducible FAIR data handling and can be adapted for deeper domain analysis or machine learning pipelines.